# Chapter 7: a small image classifier and a drifting filler

400 synthetic kernel pictures, 24 by 24 pixels, drawn with numpy: sound, broken or a foreign seed. Then one filling head that drifts low. Synthetic data only; no real brewery's numbers.

What to look at: the misses on the kernels; the hour the chart signals on the filler.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

k = pd.read_csv('../data/kernels.csv')
pix = k.filter(like='p').to_numpy() / 255
fig, ax = plt.subplots(1, 6, figsize=(9, 2))
for a, i in zip(ax, range(6)):
    a.imshow(pix[i].reshape(24, 24), cmap='gray'); a.set_title(k.kind[i], fontsize=8); a.axis('off')
plt.show()

A small neural network: one hidden layer of 32 dials. 300 pictures to learn from, 100 to test.

In [ ]:
xtr, xte, ytr, yte, ktr, kte = train_test_split(pix, k.label, k.kind, test_size=100, random_state=0, stratify=k.label)
net = MLPClassifier(hidden_layer_sizes=(32,), max_iter=800, random_state=0).fit(xtr, ytr)
pred = net.predict(xte)
print('Test accuracy:', round((pred == yte).mean(), 2))
print(pd.crosstab(kte, pred, rownames=['kind'], colnames=['model says']))

## The drifting filler

Target 500 ml, lower limit 495 ml, a drift of 0.5 ml an hour: (500 - 495) / 0.5 = 10 hours to the limit. The chart plots the mean of five packs every 15 minutes. Its limits come from two in-control hours before the drift.

In [ ]:
f = pd.read_csv('../data/filler_head7.csv')
ref = f[f.hour < 0].fill_ml
run = f[f.hour >= 0].assign(block=lambda d: (d.hour * 4).astype(int))
means = run.groupby('block').fill_ml.apply(lambda x: x.iloc[:5].mean())  # five packs every 15 minutes
lcl = ref.mean() - 3 * ref.std() / np.sqrt(5)
first = means[means < lcl].index.min() / 4
print(f'Chart signals at hour {first:.2f}')
print('Lowest fill before hour 3:', f[f.hour < 3].fill_ml.min(), 'ml')

In [ ]:
plt.plot(means.index / 4, means, marker='.')
plt.axhline(lcl, color='red', label='lower control limit')
plt.axhline(495, color='black', ls='--', label='spec limit 495 ml')
plt.xlabel('Hours'); plt.ylabel('Mean fill (ml)'); plt.legend()
plt.show()